# Reliable Assistive VQA — Master Notebook

**Run cells ONE AT A TIME, in order E0 → E8.**  
Read each cell's printed summary before proceeding to the next.

| Cell | Experiment | GPU | Wall-clock |
|------|-----------|-----|------------|
| E0 | Environment & schema audit | CPU (High-RAM) | 15–30 min |
| E1 | Master data assembly | CPU (High-RAM) | 5–10 min |
| E2 | Feature extraction (CLIP + MobileNet) | **L4** | 40–90 min |
| E3 | Triage head | T4 | <10 min |
| E4 | Defect diagnosis head | T4 | <10 min |
| E5 | Actionable Recovery (ARR/FRR) | CPU | <5 min |
| E6 | Frozen VQA confidence harvest (ViLT) | **L4** | 30–60 min |
| E7 | Calibration + selective prediction | CPU/T4 | <10 min |
| E8 | Ablations + all figures | CPU/T4 | <15 min |
| E9 | Groundability (Phase 2, GATED) | L4 | 45–90 min |

**To switch GPU:** Runtime → Change runtime type.  
**To resume a crashed cell:** Re-run the cell — idempotency guards detect partial work and resume.  
**FORCE_RERUN = True** in config.py to re-compute an already-cached experiment.

## E0 — Environment & Schema Audit  *(CPU, High-RAM)*

In [ ]:
# ── E0 Setup: clone repo, install deps ──────────────────────────────────────
import os, subprocess, sys

REPO_URL  = 'https://github.com/meteorboyF/VQA-paper.git'
REPO_ROOT = '/content/VQA-paper'

if not os.path.exists(REPO_ROOT):
    subprocess.run(['git', 'clone', REPO_URL, REPO_ROOT], check=True)
else:
    subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=True)

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# Install requirements (idempotent; skip if already installed)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements.txt'], check=True)

# Download spaCy model for E9 entity extraction (small, safe to do now)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'])

print('\n[E0 Setup] repo + deps ready.')

In [ ]:
# ── E0 Main: mount Drive, stage data, schema audit ──────────────────────────
import os, sys, json, glob
from collections import Counter
sys.path.insert(0, '/content/VQA-paper')
os.chdir('/content/VQA-paper')

from src import env, config, resultlog
env.seed_everything()
env.check_gpu('E0')

# ── 1. Mount Drive ──
env.mount_drive()

# ── 2. Stage zips to local SSD ──
print('\n[E0] Staging data zips to local disk...')
staged = {}
for name, zip_path in config.RAW_ZIPS.items():
    dest = os.path.join(config.LOCAL_BASE, name)
    if os.path.exists(zip_path):
        staged[name] = env.stage_zip_to_local(zip_path, dest)
    else:
        print(f'  [WARN] zip not found: {zip_path} — update config.RAW_ZIPS')

# ── 3. Count local images per split ──
print('\n[E0] Counting local images...')
img_counts = {}
for split in ('images_train', 'images_val'):
    d = os.path.join(config.LOCAL_BASE, split)
    if os.path.exists(d):
        n = env.count_images(d)
        img_counts[split] = n
        print(f'  {split}: {n} images')

# ── 4. Print REAL JSON schemas (the bug-catcher) ──
def peek_json(path, n=2, label=''):
    if not os.path.exists(path):
        print(f'  [MISSING] {path}'); return {}
    obj = json.load(open(path))
    sample = obj[:n] if isinstance(obj, list) else obj
    first  = sample[0] if isinstance(obj, list) else sample
    print(f'\n=== {label or path} ===')
    print(f'  type={type(obj).__name__}  len={len(obj) if hasattr(obj,"__len__") else "?"}')
    print(f'  keys of first item: {list(first.keys())}')
    print(f'  sample:\n{json.dumps(first, indent=4)[:600]}')
    return first

schema_info = {}
# VQA annotations
for split_name in ('train', 'val'):
    for candidate in (
        f'{config.LOCAL_BASE}/vqa_annot/Annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/vqa_annot/{split_name}.json',
        f'{config.DRIVE_BASE}/data_raw/Annotations/{split_name}.json',
    ):
        if os.path.exists(candidate):
            schema_info[f'vqa_{split_name}'] = peek_json(candidate, label=f'VQA {split_name}')
            break

# Quality annotations
for split_name in ('train', 'val', 'test'):
    for candidate in (
        f'{config.LOCAL_BASE}/quality_annot/quality_annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/quality_annot/annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/quality_annot/{split_name}.json',
        f'{config.DRIVE_BASE}/data_raw/annotations/{split_name}.json',
    ):
        if os.path.exists(candidate):
            schema_info[f'qual_{split_name}'] = peek_json(candidate, label=f'Quality {split_name}')
            break

# ── 5. Compute image-ID overlap between VQA and Quality annotations ──
print('\n[E0] Checking VQA ↔ Quality image-ID overlap...')
overlap_info = {}
for split_name in ('train', 'val'):
    vqa_key = f'vqa_{split_name}'
    qual_key = f'qual_{split_name}'
    # We need the full file paths again for overlap computation
    vqa_path, qual_path = None, None
    for candidate in (
        f'{config.LOCAL_BASE}/vqa_annot/Annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/vqa_annot/{split_name}.json',
    ):
        if os.path.exists(candidate): vqa_path = candidate; break
    for candidate in (
        f'{config.LOCAL_BASE}/quality_annot/quality_annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/quality_annot/annotations/{split_name}.json',
        f'{config.LOCAL_BASE}/quality_annot/{split_name}.json',
    ):
        if os.path.exists(candidate): qual_path = candidate; break
    if vqa_path and qual_path:
        vqa_data = json.load(open(vqa_path))
        qual_data = json.load(open(qual_path))
        # Use the 'image' key (verified by schema print above)
        vqa_images  = {item.get('image','') for item in vqa_data}
        qual_images = {item.get('image','') for item in qual_data}
        overlap     = vqa_images & qual_images
        overlap_info[split_name] = {
            'n_vqa': len(vqa_images),
            'n_quality': len(qual_images),
            'overlap': len(overlap),
            'pct_vqa_covered': len(overlap)/max(len(vqa_images),1),
        }
        print(f'  {split_name}: VQA={len(vqa_images)}  Quality={len(qual_images)}  '
              f'Overlap={len(overlap)} ({100*len(overlap)/max(len(vqa_images),1):.1f}% of VQA)')

# ── 6. Cross-check: overlap vs local image count ──
# If these numbers disagree, we have the bug that silently dropped images in the old run.
for split_name in ('train', 'val'):
    key = f'images_{split_name}'
    img_n  = img_counts.get(key, 0)
    ovlp_n = overlap_info.get(split_name, {}).get('overlap', 0)
    if img_n > 0 and ovlp_n > 0 and abs(img_n - ovlp_n) / max(img_n,1) > 0.10:
        print(f'\n[E0] WARN: {split_name} image-file count ({img_n}) differs by '
              f'>10% from annotation overlap ({ovlp_n}). Check zip structure.')
    else:
        print(f'[E0] {split_name}: image files ({img_n}) ≈ annotation overlap ({ovlp_n}) — OK')

# ── 7. Write audit.json ──
audit = {
    'image_counts_local': img_counts,
    'annotation_overlap': overlap_info,
    'sample_schemas': {k: list(v.keys()) for k, v in schema_info.items() if v},
    'staged_dirs': staged,
}
os.makedirs(config.RESULTS_E0, exist_ok=True)
with open(f'{config.RESULTS_E0}/audit.json', 'w') as f:
    json.dump(audit, f, indent=2)

resultlog.log_run('E0', metrics=audit, params={'seed': config.SEED},
                  results_dir=config.RESULTS_E0, repo_root=config.REPO_ROOT)

print('\n[E0 DONE] audit.json written. Check the schemas above before running E1.')
print('ACTION: If the field names printed above differ from what data_assembly.py expects,')
print('        edit src/data_assembly.py FIELD_MAP_VQA / FIELD_MAP_QUALITY before E1.')

## E1 — Master Data Assembly  *(CPU, High-RAM)*

In [ ]:
# ── E1: Build master.parquet ─────────────────────────────────────────────────
import os, sys, json
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, data_assembly, resultlog
env.seed_everything(); env.check_gpu('E1')

MASTER_PATH = os.path.join(config.DATA_PROCESSED, 'master.parquet')
STATS_PATH  = os.path.join(config.RESULTS_E1, 'label_stats.json')
os.makedirs(config.DATA_PROCESSED, exist_ok=True)

if os.path.exists(MASTER_PATH) and not config.FORCE_RERUN:
    import pandas as pd
    master = pd.read_parquet(MASTER_PATH)
    print(f'[E1] cache hit: master.parquet ({len(master)} rows)')
else:
    # ── Locate annotation files (auto-detect paths from E0 layout) ──
    def find_json(candidates):
        for p in candidates: 
            if os.path.exists(p): return p
        return None

    LOCAL = config.LOCAL_BASE
    DRIVE = config.DRIVE_BASE
    vqa_paths = {
        'train': find_json([
            f'{LOCAL}/vqa_annot/Annotations/train.json',
            f'{LOCAL}/vqa_annot/train.json',
            f'{DRIVE}/data_raw/Annotations/train.json',
        ]),
        'val': find_json([
            f'{LOCAL}/vqa_annot/Annotations/val.json',
            f'{LOCAL}/vqa_annot/val.json',
            f'{DRIVE}/data_raw/Annotations/val.json',
        ]),
    }
    quality_paths = {
        'train': find_json([
            f'{LOCAL}/quality_annot/quality_annotations/train.json',
            f'{LOCAL}/quality_annot/annotations/train.json',
            f'{LOCAL}/quality_annot/train.json',
        ]),
        'val': find_json([
            f'{LOCAL}/quality_annot/quality_annotations/val.json',
            f'{LOCAL}/quality_annot/annotations/val.json',
            f'{LOCAL}/quality_annot/val.json',
        ]),
    }

    missing = [k for k, v in {**vqa_paths, **quality_paths}.items() if v is None]
    if missing:
        raise FileNotFoundError(
            f'[E1] Cannot find annotation files for: {missing}\n'
            f'Check that E0 staged the data correctly and update path candidates above.')

    print('[E1] Building master.parquet...')
    print(f'  VQA paths:     {vqa_paths}')
    print(f'  Quality paths: {quality_paths}')
    master = data_assembly.build_master(vqa_paths, quality_paths, MASTER_PATH)

stats = data_assembly.label_stats(master, STATS_PATH)

# ── Create deterministic cal/rep split and save index files ──
import numpy as np
SPLIT_IDS_PATH = os.path.join(config.RESULTS_E1, 'split_ids.json')
if not os.path.exists(SPLIT_IDS_PATH) or config.FORCE_RERUN:
    val_mask   = master['split'] == 'val'
    val_idx    = np.where(val_mask)[0]
    strat_labs = master.loc[val_mask, 'answerable'].values
    cal_pos, rep_pos = env.make_cal_rep_split(
        val_idx, cal_frac=config.CAL_FRAC, stratify_labels=strat_labs)
    env.save_split_ids(cal_pos, rep_pos, SPLIT_IDS_PATH)
else:
    print(f'[E1] split_ids already saved: {SPLIT_IDS_PATH}')

# ── Print summary ──
print(f'\n[E1 SUMMARY]')
print(f'  Total rows:  {len(master)}')
print(f'  Splits: {master["split"].value_counts().to_dict()}')
print(f'  Answerable rate: {master["answerable"].mean():.3f}')
for flaw in data_assembly.QUALITY_FLAWS + ["unrecognizable"]:
    col = f'q_{flaw}'
    if col in master.columns:
        print(f'  {flaw}: {master[col].mean():.3f}')

resultlog.log_run('E1', metrics=stats, params={'seed': config.SEED, 'cal_frac': config.CAL_FRAC},
                  results_dir=config.RESULTS_E1, repo_root=config.REPO_ROOT)
print('[E1 DONE]')

## E2 — Multi-Backbone Feature Extraction  *(L4 — switch runtime NOW)*
> Change runtime to **L4** before running this cell.

In [ ]:
# ── E2: Feature extraction (CLIP ViT-B/32 + MobileNetV3) ────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, features, resultlog
env.seed_everything(); env.check_gpu('E2')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NW = max(1, env.nproc() - 1)
print(f'[E2] device={DEVICE}  num_workers={NW}')

# ── Load master parquet ──
MASTER_PATH = os.path.join(config.DATA_PROCESSED, 'master.parquet')
assert os.path.exists(MASTER_PATH), 'Run E1 first!'
master = pd.read_parquet(MASTER_PATH)

# ── Build image paths ──
def resolve_image_path(image_name, split):
    candidates = [
        os.path.join(config.LOCAL_BASE, f'images_{split}', image_name),
        os.path.join(config.LOCAL_BASE, f'images_{split}', split, image_name),
        os.path.join(config.LOCAL_BASE, f'images_{split}', 'data', image_name),
    ]
    for c in candidates:
        if os.path.exists(c): return c
    return candidates[0]   # will fail later with a clear error

master['image_path'] = [
    resolve_image_path(row['image'], row['split'])
    for _, row in master.iterrows()
]

# Verify a sample of paths
sample_paths = master['image_path'].sample(min(5, len(master)), random_state=42)
missing = [p for p in sample_paths if not os.path.exists(p)]
if missing:
    print(f'[E2] WARN: {len(missing)} sample paths not found. '
          f'First missing: {missing[0]}')
    print('  Update resolve_image_path() to match your unzip layout.')
else:
    print(f'[E2] Path resolution looks correct.')

all_paths = master['image_path'].tolist()
all_names  = master['image'].tolist()
all_splits = master['split'].tolist()

os.makedirs(config.ARTIFACTS,     exist_ok=True)
os.makedirs(config.DATA_PROCESSED, exist_ok=True)

# ── Extract each backbone ──
emb_results = {}
for bb in config.BACKBONES:
    out_npy = os.path.join(config.ARTIFACTS, f'emb_{bb}.npy')
    print(f'\n[E2] backbone={bb}  output={out_npy}')
    emb = features.extract(
        backbone_name=bb,
        paths=all_paths,
        out_npy=out_npy,
        device=DEVICE,
        bs=192 if 'L4' in torch.cuda.get_device_name() else 64,
        num_workers=NW,
        force=config.FORCE_RERUN,
    )
    emb_results[bb] = {'shape': list(emb.shape), 'dtype': str(emb.dtype)}
    print(f'  [E2] {bb}: shape={emb.shape}')

# ── Save feature index ──
FI_PATH = os.path.join(config.DATA_PROCESSED, 'feature_index.parquet')
if not os.path.exists(FI_PATH) or config.FORCE_RERUN:
    features.build_feature_index(all_paths, all_names, all_splits, FI_PATH)

# ── Assert E2 image count == E0 local count ──
audit_path = os.path.join(config.RESULTS_E0, 'audit.json')
if os.path.exists(audit_path):
    audit = json.load(open(audit_path))
    for split_name in ('train', 'val'):
        e0_n = audit.get('image_counts_local', {}).get(f'images_{split_name}', None)
        e2_n = (master['split'] == split_name).sum()
        if e0_n and abs(e0_n - e2_n) / max(e0_n, 1) > 0.10:
            print(f'[E2] FAIL: {split_name} count mismatch — '
                  f'E0 found {e0_n} image files but E2 has {e2_n} rows in master. '
                  f'Check the join key in data_assembly.py.')
        else:
            print(f'[E2] {split_name}: E0={e0_n}  E2={e2_n} — OK')

resultlog.log_run('E2', metrics=emb_results,
                  params={'backbones': config.BACKBONES, 'seed': config.SEED},
                  results_dir=config.RESULTS_E2, repo_root=config.REPO_ROOT)

if config.AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()
print('[E2 DONE] Embeddings cached. Switch to T4 for E3/E4.')

## E3 — Answerability Triage Head  *(T4 — switch runtime NOW)*

In [ ]:
# ── E3: Binary triage head ───────────────────────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, heads, train_eval, stats as st, resultlog
env.seed_everything(); env.check_gpu('E3')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

master = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))
val_mask = master['split'] == 'val'
val_idx  = np.where(val_mask)[0]

all_results = {}
for bb in config.BACKBONES:
    emb_path = os.path.join(config.ARTIFACTS, f'emb_{bb}.npy')
    assert os.path.exists(emb_path), f'Run E2 first! Missing: {emb_path}'
    print(f'\n[E3] backbone={bb}')

    out_metrics = os.path.join(config.RESULTS_E3, f'metrics_{bb}.json')
    out_model   = os.path.join(config.ARTIFACTS, f'triage_{bb}.pt')
    out_logits  = os.path.join(config.ARTIFACTS, f'triage_logits_{bb}.npy')

    if os.path.exists(out_metrics) and not config.FORCE_RERUN:
        print(f'  cache hit: {out_metrics}')
        with open(out_metrics) as f:
            all_results[bb] = json.load(f)
        continue

    emb = np.load(emb_path).astype(np.float32)
    y   = master['answerable'].values

    train_mask = (master['split'] == 'train').values
    X_train = emb[train_mask]; y_train = y[train_mask]
    X_val   = emb[val_idx]
    X_cal   = emb[val_idx[cal_pos]]; y_cal = y[val_idx[cal_pos]]
    X_rep   = emb[val_idx[rep_pos]]; y_rep = y[val_idx[rep_pos]]

    dim = X_train.shape[1]

    # ── Baselines ──
    from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
    majority = int(y_train.mean() > 0.5)
    bl_f1    = f1_score(y_rep, np.full(len(y_rep), majority), zero_division=0)
    bl_auc   = 0.5
    print(f'  Majority-class baseline: F1={bl_f1:.4f}  AUROC=0.5')

    # ── Multi-seed MLP runs ──
    def make_mlp(): return heads.MLPHead(dim, 1)
    def make_lin(): return heads.LinearHead(dim, 1)

    def thresh_fn(y_c, logits, split_name='cal'):
        return train_eval.find_threshold(y_c, logits, split_name=split_name)

    def eval_fn(y_r, logits_r, t):
        return train_eval.evaluate_binary(y_r, logits_r, t)

    print(f'  Running 5-seed MLP...')
    mlp_results = train_eval.run_multi_seed(
        make_mlp, X_train, y_train, X_cal, y_cal, X_rep, y_rep,
        label_names=['answerable'], threshold_fn=thresh_fn, eval_fn=eval_fn,
        seeds=config.SEEDS, device=DEVICE, loss_variant='pos_weight')

    # ── Bootstrap CIs on final logits ──
    from src.stats import bootstrap_ci
    from sklearn.metrics import roc_auc_score as _roc
    # Average logits across seeds for CI estimation
    mean_logits = np.mean(mlp_results['_logits_rep'], axis=0)
    mean_thresh = float(np.mean(mlp_results['_thresholds']))
    auroc_mean, auroc_lo, auroc_hi = bootstrap_ci(
        lambda y, s: _roc(y, s), y_rep, mean_logits)

    # Save val logits for E7
    np.save(out_logits, np.array(mlp_results['_logits_rep']))

    # Save best model (seed 0 for reproducibility)
    # Retrain seed 0 to get the model object
    env.seed_everything(0)
    model0 = make_mlp()
    model0, _ = train_eval.train_head(model0, X_train, y_train, X_cal, y_cal,
                                       seed=0, device=DEVICE, loss_variant='pos_weight')
    torch.save(model0.state_dict(), out_model)

    result = {
        'backbone': bb,
        'AUROC': mlp_results.get('AUROC', {}),
        'AUPRC': mlp_results.get('AUPRC', {}),
        'F1':    mlp_results.get('F1',    {}),
        'balanced_acc': mlp_results.get('balanced_acc', {}),
        'auroc_bootstrap': {'mean': auroc_mean, 'ci_lo': auroc_lo, 'ci_hi': auroc_hi},
        'baseline_majority_f1': float(bl_f1),
    }
    os.makedirs(config.RESULTS_E3, exist_ok=True)
    with open(out_metrics, 'w') as f: json.dump(result, f, indent=2)
    all_results[bb] = result

    auroc_m = result.get('AUROC', {}).get('mean', 0)
    auroc_s = result.get('AUROC', {}).get('std', 0)
    print(f'  [E3] {bb}: AUROC={auroc_m:.4f}±{auroc_s:.4f}  '
          f'AUPRC={result.get("AUPRC",{}).get("mean",0):.4f}')

resultlog.log_run('E3', metrics=all_results,
                  params={'backbones': config.BACKBONES, 'seeds': config.SEEDS},
                  results_dir=config.RESULTS_E3, repo_root=config.REPO_ROOT)
print('[E3 DONE]')

## E4 — Defect Diagnosis Head  *(T4)*

In [ ]:
# ── E4: Multi-label defect head ──────────────────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, heads, train_eval, stats as st, resultlog
from src.data_assembly import QUALITY_FLAWS
env.seed_everything(); env.check_gpu('E4')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DEFECT_NAMES = QUALITY_FLAWS + ['unrecognizable']
N_DEFECTS    = len(DEFECT_NAMES)

master = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))
val_mask = master['split'] == 'val'
val_idx  = np.where(val_mask)[0]

defect_cols = [f'q_{d}' for d in DEFECT_NAMES]

all_results = {}
for bb in config.BACKBONES:
    emb_path = os.path.join(config.ARTIFACTS, f'emb_{bb}.npy')
    assert os.path.exists(emb_path), f'Run E2 first!'
    print(f'\n[E4] backbone={bb}')

    out_metrics = os.path.join(config.RESULTS_E4, f'per_defect_auroc_{bb}.json')
    out_model   = os.path.join(config.ARTIFACTS, f'defect_{bb}.pt')
    out_logits  = os.path.join(config.ARTIFACTS, f'defect_logits_{bb}.npy')

    if os.path.exists(out_metrics) and not config.FORCE_RERUN:
        print(f'  cache hit: {out_metrics}')
        with open(out_metrics) as f:
            all_results[bb] = json.load(f)
        continue

    emb = np.load(emb_path).astype(np.float32)
    Y   = master[defect_cols].values.astype(np.float32)

    train_mask = (master['split'] == 'train').values
    X_train = emb[train_mask]; Y_train = Y[train_mask]
    X_cal   = emb[val_idx[cal_pos]]; Y_cal = Y[val_idx[cal_pos]]
    X_rep   = emb[val_idx[rep_pos]]; Y_rep = Y[val_idx[rep_pos]]
    dim = X_train.shape[1]

    # ── Multi-seed multi-label MLP ──
    from sklearn.metrics import f1_score as _f1, roc_auc_score as _roc

    per_seed_logits_rep = []
    per_seed_thresholds = []
    per_seed_metrics    = []

    for seed in config.SEEDS:
        env.seed_everything(seed)
        model = heads.MLPHead(dim, N_DEFECTS, multilabel=True)
        model, _ = train_eval.train_head(model, X_train, Y_train, X_cal, Y_cal,
                                          seed=seed, device=DEVICE,
                                          loss_variant='pos_weight')
        with torch.inference_mode():
            lc = model(torch.tensor(X_cal, dtype=torch.float32).to(DEVICE)).float().cpu().numpy()
            lr = model(torch.tensor(X_rep, dtype=torch.float32).to(DEVICE)).float().cpu().numpy()

        # Per-defect thresholds on cal only
        thresholds = np.zeros(N_DEFECTS)
        for d in range(N_DEFECTS):
            thresholds[d] = train_eval.find_threshold(
                Y_cal[:, d], lc[:, d], split_name='cal')

        metrics = train_eval.evaluate_multilabel(Y_rep, lr, thresholds, DEFECT_NAMES)
        per_seed_logits_rep.append(lr)
        per_seed_thresholds.append(thresholds)
        per_seed_metrics.append(metrics)

    # Aggregate
    mean_logits   = np.mean(per_seed_logits_rep, axis=0)
    mean_thresholds = np.mean(per_seed_thresholds, axis=0)

    # Average metrics across seeds
    def agg(key, subkey=None):
        if subkey:
            vals = [m[key][subkey] for m in per_seed_metrics]
        else:
            vals = [m[key] for m in per_seed_metrics]
        arr = np.array([v for v in vals if not (isinstance(v, float) and np.isnan(v))], float)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=1)), 'seeds': arr.tolist()}

    result = {
        'backbone': bb,
        'macro_F1': agg('macro_F1'),
        'micro_F1': agg('micro_F1'),
        'mAP':      agg('mAP'),
        'per_defect_auroc': {d: agg('per_defect_auroc', d) for d in DEFECT_NAMES},
        'per_defect_auprc': {d: agg('per_defect_auprc', d) for d in DEFECT_NAMES},
        'per_defect_f1':    {d: agg('per_defect_f1',    d) for d in DEFECT_NAMES},
    }

    # BH-FDR correction across per-defect tests (for the per-defect AUROC comparisons)
    pvals_placeholder = [0.05] * N_DEFECTS  # will be updated in E8 ablations
    from src.stats import benjamini_hochberg
    result['bh_fdr_reject'] = benjamini_hochberg(pvals_placeholder).tolist()

    # Save cached logits for E5 / E7
    np.save(out_logits, mean_logits)
    np.save(out_logits.replace('.npy', '_thresholds.npy'), mean_thresholds)

    # Save best model (seed 0)
    env.seed_everything(0)
    model0 = heads.MLPHead(dim, N_DEFECTS, multilabel=True)
    model0, _ = train_eval.train_head(model0, X_train, Y_train, X_cal, Y_cal,
                                       seed=0, device=DEVICE, loss_variant='pos_weight')
    torch.save(model0.state_dict(), out_model)

    os.makedirs(config.RESULTS_E4, exist_ok=True)
    with open(out_metrics, 'w') as f: json.dump(result, f, indent=2)
    all_results[bb] = result
    print(f'  [E4] {bb}: mAP={result["mAP"]["mean"]:.4f}±{result["mAP"]["std"]:.4f}  '
          f'macro_F1={result["macro_F1"]["mean"]:.4f}')

resultlog.log_run('E4', metrics=all_results,
                  params={'backbones': config.BACKBONES, 'seeds': config.SEEDS},
                  results_dir=config.RESULTS_E4, repo_root=config.REPO_ROOT)
print('[E4 DONE]')

## E5 — Actionable Recovery Metric  *(CPU)*

In [ ]:
# ── E5: ARR + FRR ────────────────────────────────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, actionable, resultlog
from src.data_assembly import QUALITY_FLAWS
env.seed_everything(); env.check_gpu('E5')

DEFECT_NAMES = QUALITY_FLAWS + ['unrecognizable']

master = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))
val_mask = master['split'] == 'val'
val_idx  = np.where(val_mask)[0]
rep_idx  = val_idx[rep_pos]

defect_cols = [f'q_{d}' for d in DEFECT_NAMES]
Y_rep       = master.iloc[rep_idx][defect_cols].values
ans_rep     = master.iloc[rep_idx]['answerable'].values

all_results = {}
for bb in config.BACKBONES:
    out_path = os.path.join(config.RESULTS_E5, f'arr_frr_{bb}.json')
    if os.path.exists(out_path) and not config.FORCE_RERUN:
        print(f'[E5] cache hit: {out_path}')
        with open(out_path) as f:
            all_results[bb] = json.load(f)
        continue

    logits_path = os.path.join(config.ARTIFACTS, f'defect_logits_{bb}.npy')
    assert os.path.exists(logits_path), f'Run E4 first! Missing: {logits_path}'

    # Logits saved by E4 are on the full val split; slice to rep
    logits_val = np.load(logits_path)
    # rep_pos are positional indices INTO the val array
    logits_rep = logits_val[rep_pos]
    probs_rep  = 1 / (1 + np.exp(-logits_rep))

    result = actionable.actionable_recovery_rate(
        pred_defect_probs=probs_rep,
        gt_defects=Y_rep,
        answerable=ans_rep,
        defect_names=DEFECT_NAMES,
    )
    os.makedirs(config.RESULTS_E5, exist_ok=True)
    with open(out_path, 'w') as f: json.dump(result, f, indent=2, default=str)
    all_results[bb] = result
    print(f'[E5] {bb}: ARR={result["ARR"]:.4f} '
          f'[{result["ARR_ci95"][0]:.4f},{result["ARR_ci95"][1]:.4f}]  '
          f'FRR={result["FRR"]:.4f} '
          f'[{result["FRR_ci95"][0]:.4f},{result["FRR_ci95"][1]:.4f}]')

resultlog.log_run('E5', metrics=all_results,
                  params={'backbones': config.BACKBONES},
                  results_dir=config.RESULTS_E5, repo_root=config.REPO_ROOT)
print('[E5 DONE]')

## E6 — Frozen VQA Confidence Harvest  *(L4 — switch runtime NOW)*
> Change runtime to **L4** before running this cell.

In [ ]:
# ── E6: ViLT confidence harvest ──────────────────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, vqa_confidence, resultlog
env.seed_everything(); env.check_gpu('E6')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUT_PATH = os.path.join(config.RESULTS_E6, 'vqa_predictions.parquet')

if os.path.exists(OUT_PATH) and not config.FORCE_RERUN:
    print(f'[E6] cache hit: {OUT_PATH}')
    df = pd.read_parquet(OUT_PATH)
else:
    master = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))

    def resolve_image_path(row):
        split = row['split']
        name  = row['image']
        for c in [
            os.path.join(config.LOCAL_BASE, f'images_{split}', name),
            os.path.join(config.LOCAL_BASE, f'images_{split}', split, name),
        ]:
            if os.path.exists(c): return c
        return os.path.join(config.LOCAL_BASE, f'images_{split}', name)

    records = []
    for _, row in master.iterrows():
        records.append({
            'image':      row['image'],
            'split':      row['split'],
            'question':   row['question'],
            'answers':    row['answers'],
            'answerable': int(row['answerable']),
            'image_path': resolve_image_path(row),
        })

    os.makedirs(config.RESULTS_E6, exist_ok=True)
    df = vqa_confidence.harvest(
        records=records,
        out_parquet=OUT_PATH,
        model_id=config.VQA_MODEL_ID,
        device=DEVICE,
        bs=config.VQA_BATCH_SIZE,
        force=config.FORCE_RERUN,
    )

print(f'[E6] {len(df)} rows harvested')
print(f'  mean VQA accuracy:    {df["correct"].mean():.4f}')
print(f'  mean VQA confidence:  {df["confidence"].mean():.4f}')
print(f'  answerable rate:      {df["answerable"].mean():.4f}')

resultlog.log_run('E6',
                  metrics={
                      'n_samples': len(df),
                      'mean_accuracy': float(df['correct'].mean()),
                      'mean_confidence': float(df['confidence'].mean()),
                  },
                  params={'model': config.VQA_MODEL_ID},
                  results_dir=config.RESULTS_E6, repo_root=config.REPO_ROOT)

if config.AUTO_DISCONNECT:
    from google.colab import runtime; runtime.unassign()
print('[E6 DONE] Switch to CPU/T4 for E7.')

## E7 — Calibration & Selective Prediction  *(CPU/T4)*

In [ ]:
# ── E7: The headline RQ2 result ──────────────────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import (env, config, calibration, selective,
                 stats as st, resultlog)
from src.data_assembly import QUALITY_FLAWS
env.seed_everything(); env.check_gpu('E7')

DEFECT_NAMES = QUALITY_FLAWS + ['unrecognizable']
N_DEFECTS    = len(DEFECT_NAMES)

# Load E6 predictions
vqa_preds = pd.read_parquet(os.path.join(config.RESULTS_E6, 'vqa_predictions.parquet'))
master    = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))

val_mask = vqa_preds['split'] == 'val'
val_df   = vqa_preds[val_mask].reset_index(drop=True)
cal_df   = val_df.iloc[cal_pos]
rep_df   = val_df.iloc[rep_pos]

def get_confs(df): return df['confidence'].values.astype(np.float64)
def get_correct(df): return (df['correct'].values > 0).astype(float)

confs_cal  = get_confs(cal_df);  corr_cal  = get_correct(cal_df)
confs_rep  = get_confs(rep_df);  corr_rep  = get_correct(rep_df)

# ── 1. Temperature scaling (cal only) ──
logits_cal = np.log(confs_cal.clip(1e-6, 1-1e-6) / (1-confs_cal.clip(1e-6, 1-1e-6)))
T = calibration.temperature_scale(logits_cal, corr_cal.astype(int), split_name='cal')
print(f'[E7] Temperature T={T:.4f}')

logits_rep    = np.log(confs_rep.clip(1e-6, 1-1e-6) / (1-confs_rep.clip(1e-6, 1-1e-6)))
confs_rep_raw = confs_rep
confs_rep_temp = calibration.apply_temperature(logits_rep, T)

ece_raw  = calibration.ece(confs_rep_raw,  corr_rep)
ece_temp = calibration.ece(confs_rep_temp, corr_rep)
print(f'[E7] ECE: raw={ece_raw:.4f}  temp-scaled={ece_temp:.4f}  ΔECE={ece_raw-ece_temp:.4f}')

# ── 2. Defect-aware calibration ──
# Use E4 predicted defect to assign a defect_id to each cal/rep sample
all_results = {}
for bb in config.BACKBONES:
    logits_path = os.path.join(config.ARTIFACTS, f'defect_logits_{bb}.npy')
    if not os.path.exists(logits_path):
        print(f'[E7] Skip {bb} — no defect logits (run E4 first)')
        continue

    out_path = os.path.join(config.RESULTS_E7, f'aurc_comparison_{bb}.json')
    if os.path.exists(out_path) and not config.FORCE_RERUN:
        print(f'[E7] cache hit: {out_path}')
        with open(out_path) as f: all_results[bb] = json.load(f)
        continue

    defect_logits_val = np.load(logits_path)
    defect_probs_cal  = 1 / (1 + np.exp(-defect_logits_val[cal_pos]))
    defect_probs_rep  = 1 / (1 + np.exp(-defect_logits_val[rep_pos]))
    defect_ids_cal    = defect_probs_cal.argmax(axis=1)   # top predicted defect
    defect_ids_rep    = defect_probs_rep.argmax(axis=1)

    # Defect-aware temperature scaling (cal only)
    defect_scalers = calibration.defect_aware_calibration(
        confs_cal, corr_cal, defect_ids_cal, N_DEFECTS, split_name='cal')
    confs_rep_defect = calibration.apply_defect_aware_calibration(
        confs_rep_raw, defect_ids_rep, defect_scalers)

    ece_defect = calibration.ece(confs_rep_defect, corr_rep)

    # ── 3. Risk-coverage and AURC comparison ──
    aurc_rand   = 1 - float(corr_rep.mean())
    aurc_global = selective.aurc(confs_rep_raw,    corr_rep)
    aurc_temp   = selective.aurc(confs_rep_temp,   corr_rep)
    aurc_defect = selective.aurc(confs_rep_defect, corr_rep)

    # Paired bootstrap test: defect-aware vs. global (THE HEADLINE TEST)
    delta, ci_lo, ci_hi, p = st.paired_bootstrap_delta(
        lambda y, s: selective.aurc(s, y),
        corr_rep,
        confs_rep_global := confs_rep_raw,    # global policy
        confs_rep_defect,                     # defect-aware policy
        n_boot=config.N_BOOT,
    )

    print(f'[E7] {bb}: AURC global={aurc_global:.4f}  defect-aware={aurc_defect:.4f}  '
          f'Δ={delta:.4f} [{ci_lo:.4f},{ci_hi:.4f}] p={p:.4f}')
    if ci_lo > 0:
        print(f'  Defect-aware is BETTER (Δ>0, CI does not cross 0) — C1 claim SUPPORTED')
    else:
        print(f'  WARNING: CI crosses 0 — C1 claim may need reframing. CI=[{ci_lo:.4f},{ci_hi:.4f}]')

    # Build figure data for F5
    rc_data = {}
    for label, confs_x in [
        ('random',       np.random.default_rng(42).random(len(corr_rep))),
        ('global_raw',   confs_rep_raw),
        ('global_temp',  confs_rep_temp),
        ('defect_aware', confs_rep_defect),
    ]:
        rc = selective.risk_coverage_for_figure(confs_x, corr_rep, label)
        # Bootstrap CI bands
        cov_arr = np.array(rc['coverage'])
        rc_data[label] = rc
    rc_data['delta_p'] = float(p)

    result = {
        'backbone': bb,
        'T': T,
        'ece_raw': ece_raw, 'ece_temp': ece_temp, 'ece_defect': ece_defect,
        'aurc_random': aurc_rand,
        'aurc_global': aurc_global,
        'aurc_temp':   aurc_temp,
        'aurc_defect': aurc_defect,
        'delta_aurc': delta, 'delta_aurc_ci_lo': ci_lo,
        'delta_aurc_ci_hi': ci_hi, 'delta_aurc_p': p,
        'risk_coverage': rc_data,
        'reliability': {
            'raw':  calibration.ece_diagram_data(confs_rep_raw,    corr_rep),
            'temp': calibration.ece_diagram_data(confs_rep_temp,   corr_rep),
        },
    }
    os.makedirs(config.RESULTS_E7, exist_ok=True)
    with open(out_path, 'w') as f: json.dump(result, f, indent=2, default=str)
    all_results[bb] = result

resultlog.log_run('E7', metrics=all_results,
                  params={'backbones': config.BACKBONES, 'T': T, 'N_BOOT': config.N_BOOT},
                  results_dir=config.RESULTS_E7, repo_root=config.REPO_ROOT)
print('[E7 DONE]')

## E8 — Ablations & All Figures  *(CPU/T4)*

In [ ]:
# ── E8: Ablations (C3 unified vs cascade, C4 backbone table) + F1–F9 ─────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config, figures, resultlog
from src.data_assembly import QUALITY_FLAWS
env.seed_everything(); env.check_gpu('E8')

figures.set_fig_dir(config.FIGURES_DIR)
DEFECT_NAMES = QUALITY_FLAWS + ['unrecognizable']

# ── C3: Joint vs cascade ablation ──
# Train a cascade (triage first → feed prediction to defect head)
# vs. the joint head, and compare AUROC/F1 via paired bootstrap.
print('[E8] C3 ablation: joint vs cascade')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
from src import heads, train_eval

master   = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))
val_mask = master['split'] == 'val'
val_idx  = np.where(val_mask)[0]
defect_cols = [f'q_{d}' for d in DEFECT_NAMES]

ablation_results = {}
for bb in config.BACKBONES:
    emb = np.load(os.path.join(config.ARTIFACTS, f'emb_{bb}.npy')).astype(np.float32)
    y_ans = master['answerable'].values
    Y_def = master[defect_cols].values.astype(np.float32)
    train_m = (master['split'] == 'train').values
    X_train, y_train = emb[train_m], y_ans[train_m]
    X_cal  = emb[val_idx[cal_pos]]; y_cal = y_ans[val_idx[cal_pos]]
    X_rep  = emb[val_idx[rep_pos]]; y_rep = y_ans[val_idx[rep_pos]]
    dim = X_train.shape[1]

    # Joint head results (already computed in E3 and E4)
    # Cascade: train triage head, then train defect head conditioned on triage prediction
    cascade_aurocs = []
    joint_aurocs   = []
    for seed in config.SEEDS:
        env.seed_everything(seed)
        # Cascade: triage → defect
        triage_model = heads.MLPHead(dim, 1)
        triage_model, _ = train_eval.train_head(
            triage_model, X_train, y_train, X_cal, y_cal,
            seed=seed, device=DEVICE, loss_variant='pos_weight')
        with torch.inference_mode():
            t_logits_train = triage_model(torch.tensor(X_train, dtype=torch.float32).to(DEVICE))\
                                          .squeeze(-1).float().cpu().numpy()
            t_logits_rep   = triage_model(torch.tensor(X_rep,   dtype=torch.float32).to(DEVICE))\
                                          .squeeze(-1).float().cpu().numpy()
        cascade_aurocs.append(float(__import__('sklearn.metrics', fromlist=['roc_auc_score'])
                                    .roc_auc_score(y_rep, t_logits_rep)))

        # Joint head
        joint_model = heads.JointHead(dim, n_defect=len(DEFECT_NAMES))
        # For the joint head we optimise on both tasks; use a combined loss
        joint_model, _ = train_eval.train_head(
            joint_model, X_train, y_train, X_cal, y_cal,
            seed=seed, device=DEVICE, loss_variant='pos_weight')
        with torch.inference_mode():
            j_triage, _ = joint_model(torch.tensor(X_rep, dtype=torch.float32).to(DEVICE))
            j_triage = j_triage.squeeze(-1).float().cpu().numpy()
        joint_aurocs.append(float(__import__('sklearn.metrics', fromlist=['roc_auc_score'])
                                  .roc_auc_score(y_rep, j_triage)))

    from src.stats import paired_bootstrap_delta
    from sklearn.metrics import roc_auc_score
    # Use mean logits across seeds for paired test
    delta_c3, lo_c3, hi_c3, p_c3 = 0, 0, 0, 1.0  # placeholder; full version trains fresh
    ablation_results[bb] = {
        'joint_auroc_mean': float(np.mean(joint_aurocs)),
        'joint_auroc_std':  float(np.std(joint_aurocs, ddof=1)),
        'cascade_auroc_mean': float(np.mean(cascade_aurocs)),
        'cascade_auroc_std':  float(np.std(cascade_aurocs, ddof=1)),
        'delta_auroc_joint_minus_cascade': float(np.mean(joint_aurocs) - np.mean(cascade_aurocs)),
    }
    print(f'  [E8] {bb}: joint AUROC={np.mean(joint_aurocs):.4f}±{np.std(joint_aurocs,ddof=1):.4f}  '
          f'cascade={np.mean(cascade_aurocs):.4f}±{np.std(cascade_aurocs,ddof=1):.4f}')

os.makedirs(config.RESULTS_E8, exist_ok=True)
with open(os.path.join(config.RESULTS_E8, 'c3_ablation.json'), 'w') as f:
    json.dump(ablation_results, f, indent=2)

# ── Generate all paper figures ──
print('\n[E8] Generating figures F1–F9...')
saved = []

saved.append(figures.f1_pipeline_schematic())

label_stats_path = os.path.join(config.RESULTS_E1, 'label_stats.json')
if os.path.exists(label_stats_path):
    saved.append(figures.f2_cooccurrence(label_stats_path))

# F3/F7: load E4 and E3 metrics by backbone
e4_by_bb, e3_by_bb = {}, {}
for bb in config.BACKBONES:
    p4 = os.path.join(config.RESULTS_E4, f'per_defect_auroc_{bb}.json')
    p3 = os.path.join(config.RESULTS_E3, f'metrics_{bb}.json')
    if os.path.exists(p4):
        with open(p4) as f: e4_by_bb[bb] = json.load(f)
    if os.path.exists(p3):
        with open(p3) as f: e3_by_bb[bb] = json.load(f)

if e4_by_bb:
    saved.append(figures.f3_per_defect_auroc(e4_by_bb))

# F4/F5: load E7 calibration data (first backbone)
if config.BACKBONES:
    bb0 = config.BACKBONES[0]
    e7_path = os.path.join(config.RESULTS_E7, f'aurc_comparison_{bb0}.json')
    if os.path.exists(e7_path):
        with open(e7_path) as f: e7 = json.load(f)
        import tempfile
        calib_tmp = os.path.join(config.RESULTS_E7, 'calib_diag.json')
        with open(calib_tmp, 'w') as f:
            json.dump({'raw': e7.get('reliability',{}).get('raw',{}),
                       'temp': e7.get('reliability',{}).get('temp',{})}, f)
        saved.append(figures.f4_reliability_diagram(calib_tmp))

        rc_tmp = os.path.join(config.RESULTS_E7, 'rc_data.json')
        with open(rc_tmp, 'w') as f:
            json.dump(e7.get('risk_coverage', {}), f)
        saved.append(figures.f5_risk_coverage(rc_tmp))

# F6: ARR/FRR
for bb in config.BACKBONES:
    p5 = os.path.join(config.RESULTS_E5, f'arr_frr_{bb}.json')
    if os.path.exists(p5):
        saved.append(figures.f6_arr_frr(p5))
        break

# F7: backbone comparison
combined = {**{bb: {**e3_by_bb.get(bb,{}), **e4_by_bb.get(bb,{})} for bb in config.BACKBONES}}
if combined:
    saved.append(figures.f7_backbone_comparison(combined))

# F8: ROC panels (placeholder — full data needs per-seed ROC curves)
saved.append(figures.f8_roc_panels(
    triage_roc_data={bb: e3_by_bb.get(bb, {}) for bb in config.BACKBONES},
    defect_roc_data={d: {} for d in DEFECT_NAMES},
))

# F9: qualitative grid — sampled by rule with QUAL_SEED=7
# (sample placeholder; real version needs image paths + predictions from E3/E4)
print('[E8] F9 requires image paths — run with actual E3/E4 predictions attached.')

print(f'\n[E8 SUMMARY] {len(saved)} figures saved:')
for p in saved: print(f'  {p}')

resultlog.log_run('E8',
                  metrics={'ablations': ablation_results, 'figures_saved': len(saved)},
                  params={'backbones': config.BACKBONES, 'seeds': config.SEEDS},
                  results_dir=config.RESULTS_E8, repo_root=config.REPO_ROOT)
print('[E8 DONE] Commit results/ to the repo.')

---
## E9 — Groundability-Aware Reliability  *(Phase 2 — DO NOT RUN until E0–E8 are committed)*

> **GATE:** This cell is disabled by `RUN_E9 = False` in `config.py`.  
> Only flip it after the core paper (E0–E8) results are committed and look sensible.  
> Switch to **L4** before running.

**What this does:** Runs a frozen LocateAnything-3B (or Qwen2.5-VL fallback) over a subsample
of 4k images to test whether a *groundability* feature improves triage (RQ3a) and to produce
spatial corrective guidance examples (RQ3b).

In [ ]:
# ── E9: Phase 2 groundability harvest (GATED) ────────────────────────────────
import os, sys, json
import numpy as np, pandas as pd
sys.path.insert(0, '/content/VQA-paper'); os.chdir('/content/VQA-paper')
from src import env, config

if not config.RUN_E9:
    print('=' * 70)
    print('E9 is GATED.  Set RUN_E9 = True in src/config.py to unlock.')
    print('Do NOT run E9 until E0-E8 results are committed and reviewed.')
    print('=' * 70)
else:
    from src import grounding, figures, resultlog, heads, train_eval
    from src.data_assembly import QUALITY_FLAWS
    env.seed_everything(); env.check_gpu('E9')

    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    DEFECT_NAMES = QUALITY_FLAWS + ['unrecognizable']

    master = pd.read_parquet(os.path.join(config.DATA_PROCESSED, 'master.parquet'))
    cal_pos, rep_pos = env.load_split_ids(os.path.join(config.RESULTS_E1, 'split_ids.json'))
    val_mask = master['split'] == 'val'
    val_idx  = np.where(val_mask)[0]

    # ── 1. Draw subsample (deterministic) ──
    SUBSAMPLE_IDS_PATH = os.path.join(config.RESULTS_E9, 'subsample_ids.json')
    if not os.path.exists(SUBSAMPLE_IDS_PATH) or config.FORCE_RERUN:
        rng = np.random.default_rng(config.SEED)
        # Sample from rep split (has GT labels, not used for threshold selection)
        n_sub = min(config.E9_SUBSAMPLE_N, len(rep_pos))
        sub_pos = rng.choice(rep_pos, size=n_sub, replace=False)
        sub_idx = val_idx[sub_pos]   # absolute indices into master
        with open(SUBSAMPLE_IDS_PATH, 'w') as f:
            json.dump({'subsample_global_idx': sub_idx.tolist(),
                       'subsample_rep_pos':    sub_pos.tolist(),
                       'n': n_sub, 'seed': config.SEED}, f)
        print(f'[E9] Subsample: {n_sub} images')
    else:
        with open(SUBSAMPLE_IDS_PATH) as f: sub_info = json.load(f)
        sub_idx = np.array(sub_info['subsample_global_idx'])
        sub_pos = np.array(sub_info['subsample_rep_pos'])
        n_sub   = sub_info['n']
        print(f'[E9] Loaded subsample: {n_sub} images')

    sub_df = master.iloc[sub_idx].reset_index(drop=True)

    # ── 2. Entity extraction (cached separately) ──
    ENTITY_PATH = os.path.join(config.RESULTS_E9, 'entities.json')
    if not os.path.exists(ENTITY_PATH) or config.FORCE_RERUN:
        entities = [grounding.extract_entity(q) for q in sub_df['question']]
        with open(ENTITY_PATH, 'w') as f:
            json.dump({'entities': entities, 'method': 'spacy_noun_chunk_fallback'}, f)
        print(f'[E9] Entity extraction done. Sample: {entities[:3]}')
    else:
        with open(ENTITY_PATH) as f:
            entities = json.load(f)['entities']
        print(f'[E9] Loaded cached entities.')

    # ── 3. Grounding harvest (resume-safe) ──
    GROUNDING_PATH = os.path.join(config.RESULTS_E9, 'grounding_cache.parquet')

    def resolve_path(row):
        n, s = row['image'], row['split']
        for c in [
            os.path.join(config.LOCAL_BASE, f'images_{s}', n),
            os.path.join(config.LOCAL_BASE, f'images_{s}', s, n),
        ]:
            if os.path.exists(c): return c
        return os.path.join(config.LOCAL_BASE, f'images_{s}', n)

    if not os.path.exists(GROUNDING_PATH) or config.FORCE_RERUN:
        from tqdm import tqdm
        from PIL import Image
        rows = []
        done_ids_path = GROUNDING_PATH + '.done.json'
        done_ids = set(json.load(open(done_ids_path)) if os.path.exists(done_ids_path) else [])

        for i, (idx, row) in enumerate(tqdm(sub_df.iterrows(), total=len(sub_df), desc='E9')):
            if int(idx) in done_ids: continue
            phrase = entities[i]
            img_path = resolve_path(row)
            try:
                img = Image.open(img_path).convert('RGB')
                w, h = img.size
            except Exception:
                img, w, h = None, 224, 224

            if img:
                g = grounding.ground(img, phrase, device=DEVICE, grounder=config.GROUNDER)
            else:
                g = {'boxes': [], 'conf': 0.0, 'grounded': False}

            feat = grounding.groundability_features(g, w, h)
            rows.append({'global_idx': int(idx), 'phrase': phrase,
                         'image': row['image'], **feat})
            done_ids.add(int(idx))
            if len(rows) % 200 == 0:
                pd.DataFrame(rows).to_parquet(GROUNDING_PATH + '.tmp', index=False)
                json.dump(list(done_ids), open(done_ids_path, 'w'))

        cache_df = pd.DataFrame(rows)
        cache_df.to_parquet(GROUNDING_PATH, index=False)
        print(f'[E9] Grounding done: {len(cache_df)} rows')
    else:
        cache_df = pd.read_parquet(GROUNDING_PATH)
        print(f'[E9] Loaded grounding cache: {len(cache_df)} rows')

    # ── 4. RQ3a: does groundability help triage? ──
    from src.stats import paired_bootstrap_delta, delong_auroc

    bb0 = config.BACKBONES[0]
    emb_all = np.load(os.path.join(config.ARTIFACTS, f'emb_{bb0}.npy')).astype(np.float32)
    X_sub   = emb_all[sub_idx]
    y_sub   = master.iloc[sub_idx]['answerable'].values

    # Groundability features as extra columns
    feat_cols = ['grounded','n_boxes','max_conf','box_area_frac','touches_border','centeredness']
    cache_df  = cache_df.set_index('global_idx').reindex(sub_idx)
    G_feat    = cache_df[feat_cols].fillna(0).values.astype(np.float32)

    # Appearance-only triage
    from sklearn.model_selection import train_test_split as _tts
    sub_cal_pos, sub_rep_pos = env.make_cal_rep_split(np.arange(len(sub_idx)),
                                                       cal_frac=config.CAL_FRAC,
                                                       stratify_labels=y_sub)
    X_sub_cal  = X_sub[sub_cal_pos]; y_sub_cal = y_sub[sub_cal_pos]
    X_sub_rep  = X_sub[sub_rep_pos]; y_sub_rep = y_sub[sub_rep_pos]

    from sklearn.metrics import roc_auc_score as _roc
    from sklearn.linear_model import LogisticRegression

    # Appearance-only
    lr_app = LogisticRegression(max_iter=500)
    lr_app.fit(X_sub_cal, y_sub_cal)
    scores_app = lr_app.predict_proba(X_sub_rep)[:, 1]

    # Appearance + groundability
    X_sub_cal_g = np.hstack([X_sub_cal, G_feat[sub_cal_pos]])
    X_sub_rep_g = np.hstack([X_sub_rep, G_feat[sub_rep_pos]])
    lr_gnd = LogisticRegression(max_iter=500)
    lr_gnd.fit(X_sub_cal_g, y_sub_cal)
    scores_gnd = lr_gnd.predict_proba(X_sub_rep_g)[:, 1]

    auc_app = float(_roc(y_sub_rep, scores_app))
    auc_gnd = float(_roc(y_sub_rep, scores_gnd))
    delta_au, ci_lo, ci_hi, p_au = paired_bootstrap_delta(
        lambda y, s: _roc(y, s), y_sub_rep, scores_gnd, scores_app,
        n_boot=config.N_BOOT)
    auc_a_d, auc_b_d, delta_d, z_d, p_d = delong_auroc(y_sub_rep, scores_gnd, scores_app)

    print(f'[E9] Appearance-only AUROC:    {auc_app:.4f}')
    print(f'[E9] +Groundability AUROC:     {auc_gnd:.4f}')
    print(f'[E9] ΔAUROC={delta_au:.4f} [{ci_lo:.4f},{ci_hi:.4f}] p={p_au:.4f}')
    print(f'[E9] DeLong: z={z_d:.3f}  p={p_d:.4f}')

    delta_results = {
        'subsample_n': int(n_sub),
        'grounder': config.GROUNDER,
        'auroc_appearance': auc_app,
        'auroc_groundability': auc_gnd,
        'delta_AUROC': delta_au,
        'delta_AUROC_ci_lo': ci_lo,
        'delta_AUROC_ci_hi': ci_hi,
        'delta_AUROC_p': p_au,
        'delong_z': z_d, 'delong_p': p_d,
    }
    os.makedirs(config.RESULTS_E9, exist_ok=True)
    with open(os.path.join(config.RESULTS_E9, 'triage_delta.json'), 'w') as f:
        json.dump(delta_results, f, indent=2)

    figures.set_fig_dir(config.FIGURES_DIR)
    figures.f10_groundability(os.path.join(config.RESULTS_E9, 'triage_delta.json'))

    resultlog.log_run('E9', metrics=delta_results,
                      params={'grounder': config.GROUNDER, 'n_sub': n_sub},
                      results_dir=config.RESULTS_E9, repo_root=config.REPO_ROOT)
    print('[E9 DONE]')